In [1]:
import requests
import base64
import json
import pandas as pd

In [2]:
s = requests.session()

In [3]:
headers = {
    "Host": "airquality.cpcb.gov.in",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:129.0) Gecko/20100101 Firefox/129.0",
    "Accept": "q=0.8;application/json;q=0.9",
    "Accept-Language": "en-US,en;q=0.5",
    "Accept-Encoding": "gzip, deflate, br, zstd",
    "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
    "Origin": "https://airquality.cpcb.gov.in",
    "Connection": "keep-alive",
    "Referer": "https://airquality.cpcb.gov.in/ccr/",
    "Sec-Fetch-Dest": "empty",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Site": "same-origin",
    "Priority": "u=0"
}

In [4]:
#payload Generation

def generate_payload_as_token(site_id, station_name):
    # Create the JSON object with the given site_id and station_name
    json_data = {
        "station_id": site_id,
        "station_name": station_name,
        "state": "",
        "city": "",
        "frequency": "15min",
        "dataType": "raw"
    }
    
    # Convert the JSON object to a string
    json_str = json.dumps(json_data)
    
    # Encode the string using Base64
    encoded_bytes = base64.b64encode(json_str.encode('utf-8'))
    encoded_str = encoded_bytes.decode('utf-8')
    
    # Return the Base64 encoded token
    return encoded_str


In [5]:
#Decoding the response from token to dict
def decode_token_response(token):
    try:
        # Decode the Base64 token to bytes
        decoded_bytes = base64.b64decode(token)
        
        # Convert bytes to string
        decoded_str = decoded_bytes.decode('utf-8')
        
        # Parse the JSON string to a dictionary
        json_data = json.loads(decoded_str)
        
        return json_data
    except Exception as e:
        return {"error": str(e)}

In [6]:
def api_request_maker(payload):
    resp = s.post('https://airquality.cpcb.gov.in/dataRepository/file_Path',json=payload,headers=headers)
    return resp.text

# Loading Station and site data

In [7]:
df = pd.read_csv('sites.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'sites.csv'

In [ ]:
df['payload_token'] = df.apply(lambda row: generate_payload_as_token(row['site_code'], row['name']), axis=1)

In [ ]:
df

,city,site_code,name,payload_token
0,Palwal,site_5050,"Shyam Nagar, Palwal - HSPCB",eyJzdGF0aW9uX2lkIjogInNpdGVfNTA1MCIsICJzdGF0aW...
1,Pithampur,site_1402,"Sector-2 Industrial Area, Pithampur - MPPCB",eyJzdGF0aW9uX2lkIjogInNpdGVfMTQwMiIsICJzdGF0aW...
2,Aurangabad,site_198,"More Chowk Waluj, Aurangabad - MPCB",eyJzdGF0aW9uX2lkIjogInNpdGVfMTk4IiwgInN0YXRpb2...
3,Aurangabad,site_5788,"MIDC Chilkalthana, Aurangabad - MPCB",eyJzdGF0aW9uX2lkIjogInNpdGVfNTc4OCIsICJzdGF0aW...
4,Aurangabad,site_5544,"Gurdeo Nagar, Aurangabad - BSPCB",eyJzdGF0aW9uX2lkIjogInNpdGVfNTU0NCIsICJzdGF0aW...
...,...,...,...,...
536,Jorapokhar,site_254,"Tata Stadium, Jorapokhar - JSPCB",eyJzdGF0aW9uX2lkIjogInNpdGVfMjU0IiwgInN0YXRpb2...
537,Pathardih,site_5832,"Mohalbani Ghat, Pathardih - DMC",eyJzdGF0aW9uX2lkIjogInNpdGVfNTgzMiIsICJzdGF0aW...
538,Narnaul,site_5053,"Shastri Nagar, Narnaul - HSPCB",eyJzdGF0aW9uX2lkIjogInNpdGVfNTA1MyIsICJzdGF0aW...
539,Sonipat,site_5041,"Murthal, Sonipat - HSPCB",eyJzdGF0aW9uX2lkIjogInNpdGVfNTA0MSIsICJzdGF0aW...


## api request

In [ ]:
%%time

## this will take time depening on internet speed
df['api_response'] = df.apply(lambda row: decode_token_response(api_request_maker(row['payload_token'])), axis=1)

CPU times: total: 750 ms
Wall time: 26.8 s


In [ ]:
def extract_filepaths(api_response, start_year=2016, end_year=2018):
    # Initialize a dictionary to hold the filepaths for each year
    filepaths = {str(year): None for year in range(start_year, end_year - 1, -1)}
    
    # Define the prefix
    prefix = "https://airquality.cpcb.gov.in/dataRepository/download_file?file_name="
    
    # Iterate over the data and assign filepaths to the corresponding year
    for item in api_response.get('data', []):
        year = item.get('year')
        if year in filepaths:
            filepaths[year] = prefix + item.get('filepath')
    
    return pd.Series(filepaths)

In [ ]:
df=df.join(df['api_response'].apply(lambda x: extract_filepaths(x)))

In [ ]:
df.to_csv('MasterDataFinal.csv',index=None)